<a href="https://colab.research.google.com/github/mdelgiudicechaves-art/MVP-Margem-de-Contribui-o---Trabalho-PSP8/blob/main/Gerador_mvp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gerador do MVP — Margem de projetos (estrutura do repositório + notebook principal)

**Autora:** Marina del Giudice Chaves · **Disciplina:** PSP8 · **Professor:** André Luiz Marques Serrano · **UnB — Engenharia de Produção**

Este notebook **gera o material de entrega** do MVP: cria a estrutura do repositório (enunciado, seção 5.1), grava o notebook principal
(`MVP_Margem_Pipeline_Medalhao.ipynb`), escreve README, LICENSE, `.gitignore` e `requirements.txt`, **audita a confidencialidade**
(termos proibidos, metadados e colunas sensíveis da planilha) e empacota tudo em um `.zip` pronto para o GitHub.

**Como usar:** (1) rode as células em ordem; (2) quando solicitado, envie a planilha *MVP Margem – Uso Público.xlsx*;
(3) opcional, mas recomendado: crie o segredo `TERMOS_PROIBIDOS` no Colab (ícone de chave) com os nomes que não podem vazar, separados por vírgula;
(4) baixe o `.zip` ao final. **A planilha nunca entra no pacote.**

In [ ]:
import os, sys, re, json, zlib, base64, zipfile, shutil, hashlib, datetime as dt
from pathlib import Path
EM_COLAB = "google.colab" in sys.modules
AUTORA, DISCIPLINA, PROFESSOR, INSTITUICAO = "Marina del Giudice Chaves", "PSP8", "André Luiz Marques Serrano", "Universidade de Brasília (UnB) — Engenharia de Produção"
NOME_REPO = "mvp-margem-projetos"
NOME_NB = "MVP_Margem_Pipeline_Medalhao.ipynb"
RAIZ = Path(os.environ.get("MVP_GERADOR_SAIDA", "/content" if EM_COLAB else ".")) / NOME_REPO
RAIZ.mkdir(parents=True, exist_ok=True)
print("Repositório será criado em:", RAIZ)

Repositório será criado em: /content/mvp-margem-projetos


In [ ]:
# Termos proibidos: ficam FORA do código (segredo do Colab 'TERMOS_PROIBIDOS' ou variável de ambiente)
termos = os.environ.get("TERMOS_PROIBIDOS", "")
if not termos and EM_COLAB:
    try:
        from google.colab import userdata
        termos = userdata.get("TERMOS_PROIBIDOS") or ""
    except Exception:
        termos = ""
TERMOS = [t.strip().lower() for t in termos.split(",") if t.strip()]
print(f"{len(TERMOS)} termo(s) proibido(s) carregado(s)." if TERMOS else "⚠ Nenhum termo proibido definido: a auditoria de texto será parcial. Crie o segredo TERMOS_PROIBIDOS.")

1 termo(s) proibido(s) carregado(s).


In [ ]:
# Estrutura de pastas (enunciado, seção 5.1)
PASTAS = ["notebooks", "data", "docs", "evidencias", "relatorio", "catalogo", "scripts"]
for p in PASTAS: (RAIZ / p).mkdir(exist_ok=True)
for p in ["data", "evidencias", "relatorio", "catalogo"]: (RAIZ / p / ".gitkeep").write_text("")
print(sorted(x.name for x in RAIZ.iterdir()))

['catalogo', 'data', 'docs', 'evidencias', 'notebooks', 'relatorio', 'scripts']


In [ ]:
# Notebook principal (embutido, comprimido) → notebooks/
NB_PRINCIPAL_B64 = (
    "..."
    "XN362eXP/h86SUtw"
)

nb_bytes = zlib.decompress(base64.b64decode("".join(NB_PRINCIPAL_B64)))
(RAIZ / "notebooks" / NOME_NB).write_bytes(nb_bytes)
print(f"Notebook principal gravado: notebooks/{NOME_NB} ({len(nb_bytes)/1024:.0f} KB)")

Notebook principal gravado: notebooks/MVP_Margem_Pipeline_Medalhao.ipynb (156 KB)


In [ ]:
# Arquivos de apoio: requirements, .gitignore, LICENSE e README
(RAIZ / "requirements.txt").write_text("pandas>=2.0\nnumpy\nscipy\nscikit-learn\nstatsmodels\nduckdb\npyarrow\nmatplotlib\nopenpyxl\n", encoding="utf-8")
(RAIZ / ".gitignore").write_text("""# Dados originais e derivados nunca vão para o repositório
data/*
!data/.gitkeep
*.xlsx
*.xls
*.csv.gz
*.duckdb
*.parquet
config_privada.json
.ipynb_checkpoints/
__pycache__/
.env
""", encoding="utf-8")
ANO = dt.date.today().year
(RAIZ / "LICENSE").write_text(f"""MIT License

Copyright (c) {ANO} {AUTORA}

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"),
to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense,
and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

Observação: a licença acima cobre apenas o código. Os dados de origem são de uso restrito e NÃO são distribuídos neste repositório.
""", encoding="utf-8")

README = f"""# MVP — Margem de projetos: o que explica as diferenças e onde proteger a margem

**Autora:** {AUTORA} · **Disciplina:** {DISCIPLINA} · **Professor:** {PROFESSOR} · **{INSTITUICAO}**

## Objetivo
Construir um pipeline de dados em nuvem (arquitetura medalhão Bronze → Silver → Gold, modelo dimensional em esquema estrela) para responder:
**quais características dos projetos de 2026 (área, escopo, tipo de contrato, valor, prazo e composição da equipe) mais explicam as diferenças de margem
entre eles, e o que isso indica sobre quais projetos priorizar na originação e como alocar equipe para proteger a margem?**

Perguntas de apoio: (1) onde a margem se concentra? (2) a equipe explica a margem? (3) prazo e modelo de receita mexem na margem?

## Dados
Planilha gerencial *MVP Margem — Uso Público* de uma empresa de serviços profissionais organizada por projetos. Nomes de projetos e clientes foram substituídos por
rótulos e os valores em R$ foram descaracterizados. **A planilha original não está neste repositório** (uso restrito, autorização da empresa: ver `docs/`).

## Como reproduzir
1. Abra `notebooks/{NOME_NB}` no Google Colab.
2. Envie a planilha quando solicitado (ou coloque em `Meu Drive/MVP_Margem/entrada/`).
3. Execute todas as células. Tempo aproximado: 2 a 5 minutos.
4. As saídas ficam em `Meu Drive/MVP_Margem/` (bronze, silver, gold, catalogo, evidencias, relatorio, meta).

## Arquitetura
| Camada | Conteúdo | Formato |
|---|---|---|
| Bronze | cópia fiel em texto + metadados de ingestão (arquivo, aba, hash, data) | Parquet |
| Silver | tipagem, regras R00–R12 documentadas, margem recalculada de insumos primários, rateio de despesas indiretas | Parquet |
| Gold | dimensões (tempo, área, escopo, tipo, cargo, projeto), fatos (margem mensal, equipe mensal) e mart por projeto | Parquet + DuckDB com PK/FK |

Também são gerados o **Catálogo de Dados** (`catalogo/`), a **análise de qualidade** (verificações, tratamento e reavaliação) e a **linhagem** (`meta/`).

## Resultados
Resumo executivo, tabelas e gráficos em `relatorio/relatorio_mvp.html` e `relatorio/relatorio_mvp.md` (gerados pelo notebook).
<!-- Após executar, copie aqui os 3 principais achados e os links para as figuras em evidencias/. -->

## Limitações
Janela curta (mar–out/2026), regra de rateio de despesas indiretas que condiciona parte dos resultados, análise observacional (associação, não causalidade).
Detalhes na seção 12 do notebook.

## Checklist de entrega
- [ ] Objetivo e perguntas documentados
- [ ] Origem, licença e data de coleta registradas
- [ ] Dados persistidos na nuvem (Drive/DuckDB)
- [ ] Modelo de dados e Catálogo
- [ ] Transformação e carga documentadas
- [ ] Análise de qualidade cobrindo todos os atributos
- [ ] Respostas, evidências e discussão por pergunta
- [ ] Autoavaliação
- [ ] Capturas de tela da execução e do Drive em `evidencias/`
- [ ] Autorização da empresa registrada em `docs/`
- [ ] Repositório conferido em janela anônima, sem a planilha original

## Licença
Código sob MIT (ver `LICENSE`). Dados não distribuídos.
"""
(RAIZ / "README.md").write_text(README, encoding="utf-8")
(RAIZ / "docs" / "AUTORIZACAO_EMPRESA.md").write_text("""# Autorização da empresa para divulgação

> **Pendente.** Registre aqui a data, o responsável e a forma da autorização (e-mail ou termo) concedida pela empresa para divulgar a versão descaracterizada dos dados
> e dos resultados. **Não** inclua o nome da empresa neste arquivo se o repositório for público: arquive o comprovante fora do repositório.
""", encoding="utf-8")
(RAIZ / "scripts" / "README.md").write_text("Reservado para scripts auxiliares (opcional). O pipeline completo está no notebook principal.\n", encoding="utf-8")
print("Arquivos de apoio gravados.")

Arquivos de apoio gravados.


## Auditoria da planilha (opcional, recomendada antes de qualquer publicação)
Envie a planilha para o Colab (ou informe o caminho em `XLSX`). A auditoria **não imprime** nomes, valores nem metadados: apenas indica se há risco.

In [ ]:
XLSX = os.environ.get("MVP_XLSX", "")
if not XLSX and EM_COLAB:
    from google.colab import files
    up = files.upload()
    XLSX = next(iter(up), "")
print("Planilha:", Path(XLSX).name if XLSX else "(nenhuma — auditoria da planilha ignorada)")

Saving MVP Margem - Uso Público.xlsx to MVP Margem - Uso Público.xlsx
Planilha: MVP Margem - Uso Público.xlsx


In [ ]:
auditoria = []
if XLSX and Path(XLSX).exists():
    with zipfile.ZipFile(XLSX) as z:
        nomes = z.namelist()
        core = z.read("docProps/core.xml").decode("utf-8", "ignore").lower() if "docProps/core.xml" in nomes else ""
        app = z.read("docProps/app.xml").decode("utf-8", "ignore").lower() if "docProps/app.xml" in nomes else ""
        textos = " ".join(z.read(n).decode("utf-8", "ignore").lower() for n in nomes if n.endswith(".xml") and ("sharedStrings" in n or n.startswith("xl/worksheets/")))
        wbxml = z.read("xl/workbook.xml").decode("utf-8", "ignore").lower() if "xl/workbook.xml" in nomes else ""
    meta_txt = core + " " + app
    for rotulo, tag in [("criador", "dc:creator"), ("última modificação por", "cp:lastmodifiedby"), ("empresa (app.xml)", "company")]:
        m = re.search(rf"<{tag}>(.*?)</{tag}>", meta_txt)
        auditoria.append((f"metadado '{rotulo}' preenchido", "sim — NÃO publique o .xlsx" if m and m.group(1).strip() else "não", "info"))
    hit_meta = [t for t in TERMOS if t in meta_txt]
    auditoria.append(("termo proibido nos metadados do arquivo", "ENCONTRADO — NÃO publique o .xlsx" if hit_meta else "não encontrado", "crit" if hit_meta else "ok"))
    hit_wb = [t for t in TERMOS if t in wbxml]
    auditoria.append(("termo proibido em caminho embutido (workbook.xml)", "ENCONTRADO — NÃO publique o .xlsx" if hit_wb else "não encontrado", "crit" if hit_wb else "ok"))
    hit_txt = [t for t in TERMOS if t in textos]
    auditoria.append(("termo proibido no conteúdo das abas", "ENCONTRADO — revisar descaracterização" if hit_txt else "não encontrado", "crit" if hit_txt else "ok"))
    auditoria.append(("coluna 'Operação do Projeto' presente", "sim — o pipeline a descarta na Bronze; não publique o .xlsx" if "operação do projeto" in textos or "operacao do projeto" in textos else "não", "info"))
    auditoria.append(("aba 'Legenda Nomes' presente", "SIM — remova antes de qualquer uso" if "legenda nomes" in textos else "não", "crit" if "legenda nomes" in textos else "ok"))
import pandas as pd
AUD = pd.DataFrame(auditoria, columns=["verificação", "resultado", "nível"]) if auditoria else pd.DataFrame(columns=["verificação", "resultado", "nível"])
display(AUD)

,verificação,resultado,nível
0,metadado 'criador' preenchido,não,info
1,metadado 'última modificação por' preenchido,sim — NÃO publique o .xlsx,info
2,metadado 'empresa (app.xml)' preenchido,não,info
3,termo proibido nos metadados do arquivo,ENCONTRADO — NÃO publique o .xlsx,crit
4,termo proibido em caminho embutido (workbook.xml),ENCONTRADO — NÃO publique o .xlsx,crit
5,termo proibido no conteúdo das abas,não encontrado,ok
6,coluna 'Operação do Projeto' presente,sim — o pipeline a descarta na Bronze; não pub...,info
7,aba 'Legenda Nomes' presente,não,ok


In [ ]:
# Auditoria dos arquivos gerados (texto e notebooks)
achados = []
for p in RAIZ.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".md", ".txt", ".ipynb", ".py", ".csv", ".html", ""} :
        txt = p.read_text(encoding="utf-8", errors="ignore").lower()
        achados += [(str(p.relative_to(RAIZ)), t) for t in TERMOS if t in txt]
if not TERMOS: print("⚠ Sem termos definidos: varredura de texto não executada.")
else: print("Varredura de texto:", "nenhum termo proibido encontrado." if not achados else f"ENCONTRADO(S): {achados}")
assert not achados, "Termo proibido nos arquivos gerados — corrija antes de empacotar"
proibidos_pacote = [p for p in RAIZ.rglob("*") if p.suffix.lower() in {".xlsx", ".xls", ".duckdb", ".parquet"}]
assert not proibidos_pacote, f"Arquivos de dados no pacote: {proibidos_pacote}"
print("Nenhum arquivo de dados no pacote.")

Varredura de texto: nenhum termo proibido encontrado.
Nenhum arquivo de dados no pacote.


In [ ]:
# Empacotamento
ZIP = RAIZ.parent / f"{NOME_REPO}.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RAIZ.rglob("*")):
        if p.is_file(): z.write(p, p.relative_to(RAIZ.parent))
print("Pacote:", ZIP, f"({ZIP.stat().st_size/1024:.0f} KB)")
print("\nPróximos passos: 1) rodar notebooks/" + NOME_NB + " no Colab; 2) preencher docs/AUTORIZACAO_EMPRESA.md; 3) tirar os prints para evidencias/; 4) git init, commit e push (o .gitignore já exclui dados).")
if EM_COLAB:
    from google.colab import files
    files.download(str(ZIP))

Pacote: /content/mvp-margem-projetos.zip (50 KB)

Próximos passos: 1) rodar notebooks/MVP_Margem_Pipeline_Medalhao.ipynb no Colab; 2) preencher docs/AUTORIZACAO_EMPRESA.md; 3) tirar os prints para evidencias/; 4) git init, commit e push (o .gitignore já exclui dados).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>